In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

In [ ]:
# Variables path
layer_script = "block"

#process=""
disco="g"

# subj = "sub-A2004"

#subj = sys.argv[1] ## name of participant list

# Carpeta general
datadir = Path(f"{disco}:\\PROYECTO_SELF")
# #carpetas generales de datos
resting_dir = datadir /"Data_self"/"Data_resting"
fif_data = resting_dir /"fif_data" 
resting_dir.mkdir(parents=True, exist_ok=True)
fif_data.mkdir(parents=True, exist_ok=True)



#carpeta analysis
output_analysis= resting_dir / "output_analysis"

acw_analysis_path = output_analysis / "acw"
acw_analysis_path.mkdir(parents=True, exist_ok=True)

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"

preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)


mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

In [3]:
file="S02b_resting_close_Condition 1_DMN (3).generic"



In [4]:
from pathlib import Path
import pandas as pd

# Ruta al archivo
file = "S02b_resting_close_Condition 1_DMN (3).generic"
generic_file_path = resting_dir / file

# Intentar leer con delimitador automático
with open(generic_file_path, 'r', encoding='utf-8') as f:
    for _ in range(5):  # Muestra las primeras líneas
        print(f.readline())

BESA Generic Data v1.1

nChannels=12

sRate=500.00

nSamples=628157

format=float



In [8]:
# Ver las primeras 20 líneas del archivo para entender qué hay antes de los datos reales
with open(generic_file_path, 'r', encoding='utf-8') as f:
    for i in range(25):
        print(f"{i+1:02d}: {f.readline().strip()}")

01: BESA Generic Data v1.1
02: nChannels=12
03: sRate=500.00
04: nSamples=628157
05: format=float
06: file=S02b_resting_close_Condition 1_DMN (3).dat
07: prestimulus=2000.000
08: epochs=157
09: baselineStart=0.000
10: baselineEnd=0.000
11: epochLength=4000.000
12: Padding=2000.000
13: conditionName=Condition 1
14: channelUnits=PCC nAm
15: channelUnits=mPFC nAm
16: channelUnits=LAG nAm
17: channelUnits=RAG nAm
18: channelUnits=LLatTemp nAm
19: channelUnits=RLatTemp nAm
20: channelUnits=NoiseLOcc nAm
21: channelUnits=NoiseROcc nAm
22: channelUnits=NoiseLFr nAm
23: channelUnits=NoiseRFr nAm
24: channelUnits=NoiseMFr nAm
25: channelUnits=NoiseMPar nAm


In [10]:
generic_file_path

WindowsPath('g:/PROYECTO_SELF/Data_self/Data_resting/S02b_resting_close_Condition 1_DMN (3).generic')

In [9]:
import numpy as np

# Cargar los datos (empezando en la línea 26)
data = np.loadtxt(generic_file_path, skiprows=25)  # (628157, 12)

# Confirmar forma esperada
assert data.shape == (157 * 4000, 12), "Dimensiones inesperadas"

# Reshape a (n_epochs, n_samples, n_channels)
data_reshaped = data.reshape(157, 4000, 12)

# Opcional: reorganizar a (n_epochs, n_channels, n_samples)
data_epochs = np.transpose(data_reshaped, (0, 2, 1))  # (157, 12, 4000)

# Listo: ahora puedes aplicar ACW sobre cada canal por época

C:\Users\UCM\AppData\Local\Temp\ipykernel_13244\3230235752.py:4: UserWarning: loadtxt: input contained no data: "g:\PROYECTO_SELF\Data_self\Data_resting\S02b_resting_close_Condition 1_DMN (3).generic"
  data = np.loadtxt(generic_file_path, skiprows=25)  # (628157, 12)


AssertionError: Dimensiones inesperadas

In [11]:
with open(generic_file_path, 'r', encoding='utf-8') as f:
    for i in range(40):
        print(f"{i+1:02d}: {f.readline().strip()}")

01: BESA Generic Data v1.1
02: nChannels=12
03: sRate=500.00
04: nSamples=628157
05: format=float
06: file=S02b_resting_close_Condition 1_DMN (3).dat
07: prestimulus=2000.000
08: epochs=157
09: baselineStart=0.000
10: baselineEnd=0.000
11: epochLength=4000.000
12: Padding=2000.000
13: conditionName=Condition 1
14: channelUnits=PCC nAm
15: channelUnits=mPFC nAm
16: channelUnits=LAG nAm
17: channelUnits=RAG nAm
18: channelUnits=LLatTemp nAm
19: channelUnits=RLatTemp nAm
20: channelUnits=NoiseLOcc nAm
21: channelUnits=NoiseROcc nAm
22: channelUnits=NoiseLFr nAm
23: channelUnits=NoiseRFr nAm
24: channelUnits=NoiseMFr nAm
25: channelUnits=NoiseMPar nAm
26: 
27: 
28: 
29: 
30: 
31: 
32: 
33: 
34: 
35: 
36: 
37: 
38: 
39: 
40: 


In [1]:
from pathlib import Path
import numpy as np

def load_besa_generic_and_dat(generic_path):
    """
    Carga archivo .dat binario usando los metadatos del archivo .generic asociado.
    
    Retorna:
        data_epochs: np.ndarray de shape (n_epochs, n_channels, n_samples)
        metadata: dict con frecuencia de muestreo, nombres de canales, etc.
    """
    # --- Leer .generic y extraer metadatos ---
    with open(generic_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = {}
    channel_names = []
    for line in lines:
        line = line.strip()
        if '=' in line:
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip()
            if key == "channelUnits":
                name = value.split()[0]
                channel_names.append(name)
            else:
                metadata[key] = value

    # Convertir metadatos clave
    n_channels = int(metadata["nChannels"])
    sfreq = float(metadata["sRate"])
    n_samples_total = int(metadata["nSamples"])
    n_epochs = int(metadata["epochs"])

    # Derivar muestras por época
    n_samples_per_epoch = n_samples_total // n_epochs

    # --- Cargar archivo .dat asociado ---
    dat_path = generic_path.with_suffix('.dat')
    data = np.fromfile(dat_path, dtype='<f4')  # float32, little-endian

    expected_size = n_epochs * n_channels * n_samples_per_epoch
    assert data.size == expected_size, (
        f"Tamaño inesperado: esperado {expected_size}, encontrado {data.size}"
    )

    # --- Dar forma a la matriz: (epochs, channels, samples) ---
    data_epochs = data.reshape(n_epochs, n_samples_per_epoch, n_channels).transpose(0, 2, 1)

    return data_epochs, {
        "sfreq": sfreq,
        "n_channels": n_channels,
        "n_epochs": n_epochs,
        "n_samples_per_epoch": n_samples_per_epoch,
        "channel_names": channel_names
    }

# 🧪 USO
generic_path = Path("G:/PROYECTO_SELF/Data_self/Data_resting/S02b_resting_close_Condition 1_DMN (3).generic")
data_epochs, info = load_besa_generic_and_dat(generic_path)

print("Forma de los datos:", data_epochs.shape)  # (157, 12, 4001)
print("Nombres de canales:", info["channel_names"])


Forma de los datos: (157, 12, 4001)
Nombres de canales: ['PCC', 'mPFC', 'LAG', 'RAG', 'LLatTemp', 'RLatTemp', 'NoiseLOcc', 'NoiseROcc', 'NoiseLFr', 'NoiseRFr', 'NoiseMFr', 'NoiseMPar']
